# Assignment 2: FedAvg and first-round LOSS

Team 08. Assignment lead: Chetan Kodand Reddy Madadi. Team members: Bharat Govil, Naveen Jarpla, Vishnu Fnu. This notebook calls the same implementation as the Slurm campaign. Reuse the frozen data; 28 configurations over three seeds give 84 runs and 1,785 first-round models. See README.md for installation, execution and results.


In [ ]:
import os, sys, subprocess
from pathlib import Path
ASSIGNMENT_DIR = Path.cwd()
assert (ASSIGNMENT_DIR / "final_campaign.py").exists(), "Open from assignment-2"
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/work/pi_juanzhai_umass_edu/chetan/684/assignment-2-20261011"))
DATA_DIR = ASSIGNMENT_DIR / "data"
PARTITIONS_ROOT = DATA_DIR / "final_protocol"
# A fresh /work output preserves the completed campaign. Override to resume it.
CAMPAIGN_ROOT = Path(os.environ.get("CAMPAIGN_ROOT", str(RUN_ROOT / "notebook-final-reproduction")))
def run(script, *args, compute=True):
    if compute and "SLURM_JOB_ID" not in os.environ:
        raise RuntimeError("On Unity, use a Slurm allocation for training and model audits")
    subprocess.run([sys.executable, str(ASSIGNMENT_DIR / script), *map(str,args)], check=True)


## Frozen dataset
The 9,000-record CSV, public-background preprocessor, schema and hashes are included. Generation is unnecessary. `generate_data.py` contains the adapted Assignment 1 generator for provenance/reproduction; it needs a GPU allocation. Do not regenerate the dataset for this experiment.


In [ ]:
import pandas as pd
from IPython.display import display
records = pd.read_csv(DATA_DIR / "synthetic_hospital_records.csv")
assert len(records) == 9000 and records.record_id.is_unique
display(records.head())


## One final-protocol setting and matching attacks
Use a CPU allocation. Index 0 selects K=1/B=32/E=1/initial-LR=0.05, seed 684; indices 0–83 cover the complete grid. The wrapper reconstructs the first round, verifies its aggregated metrics, then calls the unchanged revision-3 training implementation. Repeating resumes compatible checkpoints and respects 300 total rounds. For efficient reuse of the 12 confirmed baselines, see README.md `--prepare`; it is optional for a fresh full reproduction.

Global members: 3,600 optimization records; global nonmembers: 900 validation + 4,500 test. Local nonmembers: all records outside the given client optimization partition.


In [ ]:
run("final_campaign.py", "--data-dir", DATA_DIR,
    "--partitions-root", PARTITIONS_ROOT,
    "--output", CAMPAIGN_ROOT, "--run-index", 0)


## Independent audit
After all 84 indices finish, this rescoring audit requires a Slurm CPU allocation. It validates 1,785 first-round and 168 selected/last models and exports all tables under `analysis/results`. Early stops and caps remain separate from loss-stable endpoints. Oracle-threshold membership BA uses labels retrospectively; classifier thresholds use validation labels.


In [ ]:
run("final_campaign.py", "--data-dir", DATA_DIR,
    "--partitions-root", PARTITIONS_ROOT,
    "--output", CAMPAIGN_ROOT, "--audit")


## Review the completed local submission
These tables come from the audited final campaign. Historical trials are archived outside the submission as explained in README.md. Copy new audited tables only after successful coverage checks. `render_report.py` rebuilds figures and LaTeX from the exported CSVs and preserved historical provenance; compile and run `check_pdfs.py` as documented in README.md.


In [ ]:
display(pd.read_csv(ASSIGNMENT_DIR / "results/run_status.csv"))
display(pd.read_csv(ASSIGNMENT_DIR / "results/attack_summary.csv"))
run("render_report.py", "--destination", ASSIGNMENT_DIR, compute=False)
